# Extracción de texto desde PDF con pdfminer.six

Este cuaderno implementa el algoritmo de extracción de texto usando la librería **pdfminer.six**.

El objetivo es extraer frases desde documentos PDF digitales, separando cada frase por punto. No se realiza OCR porque el estudio se limita a documentos PDF no escaneados.

El algoritmo genera dos archivos de salida:

1. `frases_extraidas_pdfminer.csv`: contiene las frases extraídas.
2. `metricas_pdfminer.csv`: contiene las métricas de rendimiento y calidad básica de la extracción.

La estructura de salida mantiene las mismas columnas utilizadas en el cuaderno de PyMuPDF, para facilitar la comparación posterior entre librerías.

In [ ]:
!pip install pdfminer.six

In [ ]:
import pandas as pd
import re
import time
import os
from google.colab import files
from pdfminer.high_level import extract_text
from pdfminer.pdfpage import PDFPage

## 1. Carga del archivo PDF

Seleccione el archivo PDF desde su equipo local. El archivo debe ser un PDF digital, no escaneado.

In [ ]:
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
nombre_pdf = os.path.basename(pdf_path)

print("Archivo cargado:", nombre_pdf)

## 2. Funciones auxiliares

Estas funciones limpian el texto, segmentan las frases por punto y evalúan si una frase tiene un valor mínimo para el corpus inicial.

In [ ]:
def limpiar_texto(texto):
    texto = texto.replace("\n", " ")
    texto = re.sub(r"\s+", " ", texto)
    texto = texto.strip()
    return texto


def separar_frases_por_punto(texto):
    texto = limpiar_texto(texto)
    partes = texto.split(".")

    frases = []
    for parte in partes:
        frase = parte.strip()
        if frase:
            frases.append(frase + ".")

    return frases


def contar_palabras(texto):
    palabras = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", texto)
    return len(palabras)


def es_frase_valida(frase, minimo_palabras=3, minimo_caracteres=10):
    frase_limpia = frase.strip()

    if len(frase_limpia) < minimo_caracteres:
        return False

    if contar_palabras(frase_limpia) < minimo_palabras:
        return False

    if re.fullmatch(r"[\d\W_]+", frase_limpia):
        return False

    return True

## 3. Conteo de páginas del PDF

pdfminer.six permite extraer texto por rangos de página. Primero se calcula el número total de páginas para procesarlas individualmente y conservar la columna `pagina`.

In [ ]:
def contar_paginas_pdf(pdf_path):
    with open(pdf_path, "rb") as archivo:
        total_paginas = sum(1 for _ in PDFPage.get_pages(archivo))

    return total_paginas


total_paginas_pdf = contar_paginas_pdf(pdf_path)
print("Total de páginas:", total_paginas_pdf)

## 4. Extracción de frases con pdfminer.six

El algoritmo procesa el PDF página por página, extrae el texto y separa las frases por punto. También registra si alguna página genera error durante la extracción.

In [ ]:
def extraer_frases_pdfminer(pdf_path):
    registros = []
    paginas_con_error = 0
    errores = []

    total_paginas = contar_paginas_pdf(pdf_path)

    inicio = time.time()

    for indice_pagina in range(total_paginas):
        numero_pagina = indice_pagina + 1

        try:
            texto_pagina = extract_text(pdf_path, page_numbers=[indice_pagina])
            frases_pagina = separar_frases_por_punto(texto_pagina)

            for frase in frases_pagina:
                num_palabras = contar_palabras(frase)
                num_caracteres = len(frase)

                registros.append({
                    "id": len(registros) + 1,
                    "pagina": numero_pagina,
                    "frase": frase,
                    "num_palabras": num_palabras,
                    "num_caracteres": num_caracteres,
                    "es_valida": es_frase_valida(frase)
                })

        except Exception as e:
            paginas_con_error += 1
            errores.append({
                "pagina": numero_pagina,
                "error": str(e)
            })

    fin = time.time()
    tiempo_segundos = round(fin - inicio, 4)

    df_frases = pd.DataFrame(registros)

    info_proceso = {
        "total_paginas": total_paginas,
        "tiempo_segundos": tiempo_segundos,
        "paginas_con_error": paginas_con_error,
        "errores": errores
    }

    return df_frases, info_proceso


df_frases, info_proceso = extraer_frases_pdfminer(pdf_path)

print("Extracción finalizada")
print("Total de páginas:", info_proceso["total_paginas"])
print("Tiempo de extracción:", info_proceso["tiempo_segundos"], "segundos")
print("Total de frases extraídas:", len(df_frases))

df_frases.head(20)

## 5. Cálculo de métricas

Se calculan métricas estandarizadas para comparar pdfminer.six con PyMuPDF y con las demás librerías que se implementen después.

In [ ]:
def calcular_metricas(df_frases, info_proceso, nombre_pdf, libreria="pdfminer.six"):
    if df_frases.empty:
        total_caracteres = 0
        total_palabras = 0
        total_frases = 0
        frases_validas = 0
        frases_invalidas = 0
        frases_duplicadas = 0
        porcentaje_frases_validas = 0
        porcentaje_duplicados = 0
        promedio_palabras_por_frase = 0
        promedio_caracteres_por_pagina = 0
    else:
        total_caracteres = int(df_frases["num_caracteres"].sum())
        total_palabras = int(df_frases["num_palabras"].sum())
        total_frases = int(len(df_frases))

        frases_validas = int(df_frases["es_valida"].sum())
        frases_invalidas = int(total_frases - frases_validas)

        frases_duplicadas = int(df_frases.duplicated(subset=["frase"]).sum())

        porcentaje_frases_validas = round((frases_validas / total_frases) * 100, 2) if total_frases > 0 else 0
        porcentaje_duplicados = round((frases_duplicadas / total_frases) * 100, 2) if total_frases > 0 else 0

        promedio_palabras_por_frase = round(df_frases["num_palabras"].mean(), 2)
        promedio_caracteres_por_pagina = round(
            total_caracteres / info_proceso["total_paginas"], 2
        ) if info_proceso["total_paginas"] > 0 else 0

    metricas = {
        "libreria": libreria,
        "archivo_pdf": nombre_pdf,
        "total_paginas": info_proceso["total_paginas"],
        "tiempo_segundos": info_proceso["tiempo_segundos"],
        "total_caracteres": total_caracteres,
        "total_palabras": total_palabras,
        "total_frases": total_frases,
        "frases_validas": frases_validas,
        "frases_invalidas": frases_invalidas,
        "porcentaje_frases_validas": porcentaje_frases_validas,
        "frases_duplicadas": frases_duplicadas,
        "porcentaje_duplicados": porcentaje_duplicados,
        "promedio_palabras_por_frase": promedio_palabras_por_frase,
        "promedio_caracteres_por_pagina": promedio_caracteres_por_pagina,
        "paginas_con_error": info_proceso["paginas_con_error"]
    }

    return pd.DataFrame([metricas])


df_metricas = calcular_metricas(df_frases, info_proceso, nombre_pdf)

df_metricas

## 6. Revisión rápida de frases válidas e inválidas

Esta revisión permite inspeccionar ejemplos de la extracción antes de guardar los archivos finales.

In [ ]:
print("Ejemplos de frases válidas:")
display(df_frases[df_frases["es_valida"] == True].head(10))

print("Ejemplos de frases inválidas:")
display(df_frases[df_frases["es_valida"] == False].head(10))

## 7. Guardar los archivos de salida

Se generan dos archivos CSV:

- `frases_extraidas_pdfminer.csv`
- `metricas_pdfminer.csv`

In [ ]:
archivo_frases = "frases_extraidas_pdfminer.csv"
archivo_metricas = "metricas_pdfminer.csv"

df_frases.to_csv(archivo_frases, index=False, encoding="utf-8-sig")
df_metricas.to_csv(archivo_metricas, index=False, encoding="utf-8-sig")

print("Archivo de frases generado:", archivo_frases)
print("Archivo de métricas generado:", archivo_metricas)

In [ ]:
files.download(archivo_frases)
files.download(archivo_metricas)